# Staging Customers

In [0]:
from pyspark.sql import functions as F

bronze_customers = spark.table("brazilian_ecommerce.prod.bronze_customers")

staging_customers = (
    bronze_customers
    .drop("customer_city")   # city still dropped; state is needed for gold_delivery_by_region
    .filter(F.col("customer_id").isNotNull())   # PK can't be null
    .dropDuplicates(["customer_id"])            # guarantee PK uniqueness
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_customers"

(
    staging_customers.write
    .mode("overwrite")
    .option("comment", "customers staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# enforce primary key at the table level (Unity Catalog, informational constraint)
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN customer_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_customers PRIMARY KEY (customer_id)
""")

print(f"Loaded {staging_customers.count()} rows into {full_table_name}")